# Retirada das Instâncias viciadas

In [1]:
import pandas as pd

df = pd.read_csv("resultados/instancias_NORM_e_anormal_juntos.csv")

print(f"Quantidade de registros: {len(df)}")

Quantidade de registros: 445


# Divisão dos Rotulos 0 e 1 com subcategorias

Divisão dos Rotulos 0 e 1 com subcategorias aqui , só que use essas informações da instancias_NORM_e_anormal_juntos.csv como critérios para não serem selecionadas para nenhum lugar 


In [2]:

import pandas as pd
import ast
from pathlib import Path

# ============================================================
# 1. CARREGAMENTO DAS BASES
# ============================================================

PASTA = Path.cwd()

def localizar(nome):
    arquivos = sorted(
        p for p in PASTA.rglob("*.csv")
        if p.name.lower().startswith(nome.lower())
        and "resultados" not in p.parts
    )
    if len(arquivos) != 1:
        raise FileNotFoundError(
            f"Esperado um arquivo {nome}, encontrados: {arquivos}"
        )
    return arquivos[0]

df = pd.read_csv(localizar("ptbxl_database"))
scp = pd.read_csv(localizar("scp_statements"))
scp = scp.set_index(scp.columns[0])

excluidos = pd.read_csv(
    PASTA / "resultados" / "instancias_NORM_e_anormal_juntos.csv"
)

# ============================================================
# 2. EXCLUSÃO DAS INSTÂNCIAS COM NORM + ANORMALIDADE
# ============================================================

ids_excluidos = set(excluidos["ecg_id"])

df_filtrado = df[
    ~df["ecg_id"].isin(ids_excluidos)
].copy()

print("Registros originais:", len(df))
print("Registros excluídos:", len(df) - len(df_filtrado))
print("Registros restantes:", len(df_filtrado))

# ============================================================
# 3. DEFINIÇÃO DAS SUBCATEGORIAS
# ============================================================

subcategorias = {
    "1.1": "MI",
    "1.2": "STTC",
    "1.3": "CD",
    "1.4": "HYP",
    "1.5": "FORM",
    "1.6": "RHYTHM"
}

codigos_sinusais = {"SR", "SBRAD", "STACH", "SARRH"}

def classificar(valor):

    codigos = ast.literal_eval(valor)

    categorias = set()

    for codigo in codigos:

        if codigo not in scp.index:
            continue

        info = scp.loc[codigo]

        if info["diagnostic"] == 1:
            classe = info["diagnostic_class"]

            for numero, nome in subcategorias.items():
                if classe == nome:
                    categorias.add(numero)

        if info["form"] == 1 and info["diagnostic"] != 1:
            categorias.add("1.5")

        if info["rhythm"] == 1 and codigo not in codigos_sinusais:
            categorias.add("1.6")

    if categorias:
        return pd.Series([
            1,
            "; ".join(sorted(categorias))
        ])

    if "NORM" in codigos or any(
        codigo in codigos for codigo in codigos_sinusais
    ):
        return pd.Series([0, "0"])

    return pd.Series([pd.NA, "Revisar"])

# ============================================================
# 4. CLASSIFICAÇÃO
# ============================================================

df_filtrado[["rotulo", "subcategorias"]] = (
    df_filtrado["scp_codes"].apply(classificar)
)

# ============================================================
# 5. SEPARAÇÃO DAS CLASSES
# ============================================================

classe_0 = df_filtrado[
    df_filtrado["rotulo"] == 0
].copy()

classe_1 = df_filtrado[
    df_filtrado["rotulo"] == 1
].copy()

revisar = df_filtrado[
    df_filtrado["rotulo"].isna()
].copy()

# ============================================================
# 6. RELATÓRIO
# ============================================================

print("\n===== DIVISÃO DOS RÓTULOS =====")
print("Classe 0 (Normal/Sinusal):", len(classe_0))
print("Classe 1 (Anormal):", len(classe_1))
print("Sem classificação:", len(revisar))

print("\n===== SUBCATEGORIAS ANORMAIS =====")

for numero, nome in subcategorias.items():
    quantidade = classe_1["subcategorias"].apply(
        lambda x: numero in x.split("; ")
    ).sum()

    print(f"{numero} - {nome}: {quantidade}")

# ============================================================
# 7. VALIDAÇÃO DAS EXCLUSÕES
# ============================================================

assert not set(df_filtrado["ecg_id"]) & ids_excluidos
assert not set(classe_0["ecg_id"]) & ids_excluidos
assert not set(classe_1["ecg_id"]) & ids_excluidos

print("\nValidação concluída: nenhuma instância excluída foi classificada.")


Registros originais: 21799
Registros excluídos: 445
Registros restantes: 21354

===== DIVISÃO DOS RÓTULOS =====
Classe 0 (Normal/Sinusal): 8069
Classe 1 (Anormal): 13285
Sem classificação: 0

===== SUBCATEGORIAS ANORMAIS =====
1.1 - MI: 5468
1.2 - STTC: 5202
1.3 - CD: 4483
1.4 - HYP: 2644
1.5 - FORM: 7233
1.6 - RHYTHM: 2136

Validação concluída: nenhuma instância excluída foi classificada.


In [3]:

import ast
import pandas as pd
from pathlib import Path

# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

PASTA = Path.cwd()
SAIDA = PASTA / "resultados" / "Classificacao"

SUBCATEGORIAS = {
    "1.1": "MI",
    "1.2": "STTC",
    "1.3": "CD",
    "1.4": "HYP",
    "1.5": "FORM",
    "1.6": "RHYTHM"
}

SINUSAIS = {"SR", "SBRAD", "STACH", "SARRH"}

# ============================================================
# 2. LOCALIZAÇÃO DOS ARQUIVOS
# ============================================================

def localizar_csv(prefixo, excluir_resultados=False):

    arquivos = sorted(
        p for p in PASTA.rglob("*.csv")
        if p.name.lower().startswith(prefixo.lower())
        and (
            not excluir_resultados
            or "resultados" not in p.relative_to(PASTA).parts
        )
    )

    if not arquivos:
        raise FileNotFoundError(
            f"Arquivo não encontrado: {prefixo}"
        )

    if len(arquivos) > 1:
        raise RuntimeError(
            f"Mais de um arquivo encontrado para {prefixo}:\n"
            + "\n".join(map(str, arquivos))
            + "\nInforme a versão desejada para evitar ambiguidades."
        )

    return arquivos[0]


ARQUIVO_ECG = localizar_csv(
    "ptbxl_database",
    excluir_resultados=True
)

ARQUIVO_SCP = localizar_csv(
    "scp_statements",
    excluir_resultados=True
)

# Procura primeiro o CSV de exclusões na pasta resultados.
# Aceita também nomes como (...)(1).csv.

pasta_exclusoes = PASTA / "resultados"

candidatos = sorted(
    p for p in pasta_exclusoes.glob(
        "instancias_NORM_e_anormal_juntos*.csv"
    )
    if p.is_file()
)

if len(candidatos) == 1:
    ARQUIVO_EXCLUSOES = candidatos[0]

elif len(candidatos) == 0:
    ARQUIVO_EXCLUSOES = localizar_csv(
        "instancias_NORM_e_anormal_juntos"
    )

else:
    raise RuntimeError(
        "Há vários CSVs de exclusão. "
        "Mantenha somente a versão desejada:\n"
        + "\n".join(map(str, candidatos))
    )

print("Base PTB-XL:", ARQUIVO_ECG)
print("Dicionário SCP:", ARQUIVO_SCP)
print("Tabela de exclusão:", ARQUIVO_EXCLUSOES)

# ============================================================
# 3. CARREGAMENTO
# ============================================================

df_original = pd.read_csv(ARQUIVO_ECG)
scp = pd.read_csv(ARQUIVO_SCP)
exclusoes = pd.read_csv(ARQUIVO_EXCLUSOES)

scp = scp.set_index(scp.columns[0])

# ============================================================
# 4. VALIDAÇÃO DA TABELA DE EXCLUSÃO
# ============================================================

if "ecg_id" not in exclusoes.columns:
    raise ValueError(
        "A tabela de exclusão não possui a coluna ecg_id."
    )

if exclusoes["ecg_id"].isna().any():
    raise ValueError(
        "Existem ecg_id vazios na tabela de exclusão."
    )

if exclusoes["ecg_id"].duplicated().any():
    raise ValueError(
        "Existem ecg_id duplicados na tabela de exclusão."
    )

ids_excluidos = set(
    exclusoes["ecg_id"].astype(int)
)

ids_originais = set(
    df_original["ecg_id"].astype(int)
)

if len(ids_excluidos) != 445:
    raise ValueError(
        "Esperados 445 IDs na tabela de exclusão, "
        f"mas foram encontrados {len(ids_excluidos)}."
    )

if not ids_excluidos.issubset(ids_originais):
    faltantes = ids_excluidos - ids_originais

    raise ValueError(
        "Existem IDs de exclusão ausentes na base: "
        f"{sorted(faltantes)}"
    )

print("\n===== EXCLUSÕES =====")
print("Total original:", len(df_original))
print("IDs de exclusão:", len(ids_excluidos))

# ============================================================
# 5. EXCLUSÃO ANTES DA CLASSIFICAÇÃO
# ============================================================

mascara_exclusao = df_original[
    "ecg_id"
].astype(int).isin(ids_excluidos)

df = df_original.loc[
    ~mascara_exclusao
].copy()

removidos = df_original.loc[
    mascara_exclusao
].copy()

assert len(removidos) == 445
assert not set(df["ecg_id"]) & ids_excluidos

print("Instâncias removidas:", len(removidos))
print("Instâncias restantes:", len(df))

# ============================================================
# 6. INTERPRETAÇÃO DOS CÓDIGOS
# ============================================================

def interpretar(valor):

    if isinstance(valor, dict):
        return valor

    codigos = ast.literal_eval(valor)

    if not isinstance(codigos, dict):
        raise ValueError(
            f"scp_codes inválido: {valor}"
        )

    return codigos


# ============================================================
# 7. CLASSIFICAÇÃO
# ============================================================

def classificar(valor):

    codigos = interpretar(valor)

    categorias = set()
    desconhecidos = []

    for codigo in codigos:

        if codigo not in scp.index:
            desconhecidos.append(codigo)
            continue

        info = scp.loc[codigo]

        # Superclasses diagnósticas
        if info["diagnostic"] == 1:

            classe = info["diagnostic_class"]

            for numero, nome in SUBCATEGORIAS.items():

                if classe == nome:
                    categorias.add(numero)

        # Alterações morfológicas
        if (
            info["form"] == 1
            and info["diagnostic"] != 1
        ):
            categorias.add("1.5")

        # Alterações de ritmo não sinusais
        if (
            info["rhythm"] == 1
            and codigo not in SINUSAIS
        ):
            categorias.add("1.6")

    if desconhecidos:
        return pd.Series([
            pd.NA,
            "",
            "REVISAR_CODIGO_DESCONHECIDO"
        ])

    # Alteração prevalece sobre normalidade/sinusal
    if categorias:
        return pd.Series([
            1,
            "; ".join(sorted(categorias)),
            "ANORMAL"
        ])

    # Normal ou sinusal
    if (
        "NORM" in codigos
        or bool(set(codigos) & SINUSAIS)
    ):
        return pd.Series([
            0,
            "0",
            "NORMAL_SINUSAL"
        ])

    return pd.Series([
        pd.NA,
        "",
        "REVISAR"
    ])


df[["rotulo", "subcategorias", "situacao"]] = (
    df["scp_codes"].apply(classificar)
)

# ============================================================
# 8. SEPARAÇÃO DAS CLASSES
# ============================================================

classe_0 = df[
    df["rotulo"] == 0
].copy()

classe_1 = df[
    df["rotulo"] == 1
].copy()

revisar = df[
    df["rotulo"].isna()
].copy()

# ============================================================
# 9. PREPARAR SUBCATEGORIAS
# ============================================================

dados_subcategorias = {}
contagens = {}

for numero, nome in SUBCATEGORIAS.items():

    mascara = classe_1["subcategorias"].apply(
        lambda valor: numero in valor.split("; ")
    )

    dados_sub = classe_1.loc[mascara].copy()

    dados_subcategorias[numero] = dados_sub
    contagens[numero] = len(dados_sub)

# ============================================================
# 10. VALIDAÇÃO OBRIGATÓRIA
# ============================================================

def validar_sem_excluidos(nome, dados):

    encontrados = set(
        dados["ecg_id"].astype(int)
    ) & ids_excluidos

    if encontrados:
        raise AssertionError(
            f"ERRO: {nome} contém IDs excluídos: "
            f"{sorted(encontrados)}"
        )


validar_sem_excluidos("Base filtrada", df)
validar_sem_excluidos("Classe 0", classe_0)
validar_sem_excluidos("Classe 1", classe_1)
validar_sem_excluidos("Revisar", revisar)

for numero, dados_sub in dados_subcategorias.items():

    validar_sem_excluidos(
        f"Subcategoria {numero}",
        dados_sub
    )

assert len(df_original) == len(df) + len(removidos)

assert (
    len(classe_0)
    + len(classe_1)
    + len(revisar)
    == len(df)
)

assert not (
    set(classe_0["ecg_id"])
    & set(classe_1["ecg_id"])
)

print("\nVALIDAÇÃO APROVADA")
print("Nenhuma das 445 instâncias entrou nas classes.")

# ============================================================
# 11. EXPORTAÇÃO DOS CSVs
# ============================================================

PASTA_0 = SAIDA / "0"
PASTA_1 = SAIDA / "1"
PASTA_REVISAO = SAIDA / "Revisar"

for pasta in [PASTA_0, PASTA_1, PASTA_REVISAO]:
    pasta.mkdir(parents=True, exist_ok=True)

# Sobrescreve os CSVs de classificação anteriores
classe_0.to_csv(
    PASTA_0 / "classe_0.csv",
    index=False,
    encoding="utf-8-sig"
)

classe_1.to_csv(
    PASTA_1 / "classe_1.csv",
    index=False,
    encoding="utf-8-sig"
)

revisar.to_csv(
    PASTA_REVISAO / "revisar.csv",
    index=False,
    encoding="utf-8-sig"
)

for numero, nome in SUBCATEGORIAS.items():

    pasta_sub = PASTA_1 / numero
    pasta_sub.mkdir(parents=True, exist_ok=True)

    dados_subcategorias[numero].to_csv(
        pasta_sub / f"classe_{numero}_{nome}.csv",
        index=False,
        encoding="utf-8-sig"
    )

# ============================================================
# 12. RELATÓRIO FINAL
# ============================================================

print("\n" + "=" * 55)
print("RELATÓRIO FINAL")
print("=" * 55)

print("Total original:", len(df_original))
print("Instâncias excluídas:", len(removidos))
print("Total após exclusões:", len(df))

print("\nCLASSES")

print("0 - Normal/Sinusal:", len(classe_0))
print("1 - Anormal:", len(classe_1))
print("Revisar:", len(revisar))

print("\nSUBCATEGORIAS")

for numero, nome in SUBCATEGORIAS.items():

    print(
        f"{numero} - {nome}: "
        f"{contagens[numero]}"
    )

print("\n" + "=" * 55)
print("EXCLUSÃO VERIFICADA COM SUCESSO")
print("=" * 55)

print("445 instâncias excluídas.")
print("0 instâncias excluídas na classe 0.")
print("0 instâncias excluídas na classe 1.")
print("0 instâncias excluídas nas subcategorias.")

print(f"\nArquivos salvos em: {SAIDA}")


Base PTB-XL: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\PTB-XL\ptbxl_database.csv
Dicionário SCP: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\PTB-XL\scp_statements.csv
Tabela de exclusão: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\resultados\instancias_NORM_e_anormal_juntos.csv

===== EXCLUSÕES =====
Total original: 21799
IDs de exclusão: 445
Instâncias removidas: 445
Instâncias restantes: 21354

VALIDAÇÃO APROVADA
Nenhuma das 445 instâncias entrou nas classes.

RELATÓRIO FINAL
Total original: 21799
Instâncias excluídas: 445
Total após exclusões: 21354

CLASSES
0 - Normal/Sinusal: 8069
1 - Anormal: 13285
Revisar: 0

SUBCATEGORIAS
1.1 - MI: 5468
1.2 - STTC: 5202
1.3 - CD: 4483
1.4 - HYP: 2644
1.5 - FORM: 7233
1.6 - RHYTHM: 2136

EXCLUSÃO VERIFICADA COM SUCESSO
445 instâncias excluídas.
0 instâncias excluídas na classe 0.
0 instâncias excluídas na classe 1.
0 instâncias excluídas nas subcategorias.

Arquivos salvos em: c:\Users\Matheus\Documents\Detctor-automati

# Análise da Quantidade de Instâncias Únicas por Classe e Subcategoria

In [4]:
from pathlib import Path
import pandas as pd

PASTA = Path("resultados/Classificacao")

arquivos = {
    "0 - Normal/Sinusal": PASTA / "0/classe_0.csv",
    "1 - Anormal": PASTA / "1/classe_1.csv",
    "1.1 - MI": PASTA / "1/1.1/classe_1.1_MI.csv",
    "1.2 - STTC": PASTA / "1/1.2/classe_1.2_STTC.csv",
    "1.3 - CD": PASTA / "1/1.3/classe_1.3_CD.csv",
    "1.4 - HYP": PASTA / "1/1.4/classe_1.4_HYP.csv",
    "1.5 - FORM": PASTA / "1/1.5/classe_1.5_FORM.csv",
    "1.6 - RHYTHM": PASTA / "1/1.6/classe_1.6_RHYTHM.csv"
}

resultado = []

for classe, caminho in arquivos.items():
    df_temp = pd.read_csv(caminho)

    resultado.append({
        "Classe": classe,
        "Total": len(df_temp),
        "Únicos (ecg_id)": df_temp["ecg_id"].nunique(),
        "Duplicados": df_temp["ecg_id"].duplicated().sum()
    })

resumo = pd.DataFrame(resultado)

display(resumo)

,Classe,Total,Únicos (ecg_id),Duplicados
0,0 - Normal/Sinusal,8069,8069,0
1,1 - Anormal,13285,13285,0
2,1.1 - MI,5468,5468,0
3,1.2 - STTC,5202,5202,0
4,1.3 - CD,4483,4483,0
5,1.4 - HYP,2644,2644,0
6,1.5 - FORM,7233,7233,0
7,1.6 - RHYTHM,2136,2136,0


In [6]:
import pandas as pd

df_mi = pd.read_csv(
    "resultados/Classificacao/1/1.1/classe_1.1_MI.csv"
)

colunas = [
    "ecg_id", "patient_id", "age", "sex", "height", "weight",
    "nurse", "site", "device", "recording_date", "report",
    "scp_codes", "heart_axis", "infarction_stadium1",
    "infarction_stadium2", "validated_by", "second_opinion",
    "initial_autogenerated_report", "validated_by_human",
    "baseline_drift", "static_noise", "burst_noise",
    "electrodes_problems", "extra_beats", "pacemaker",
    "strat_fold", "filename_lr", "filename_hr",
    "rotulo", "subcategorias", "situacao"
]

display(df_mi[colunas])

,ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,...,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr,rotulo,subcategorias,situacao
0,8,11275.0,48.0,0,NaN,95.0,2.0,0.0,CS-12 E,1984-12-01 14:49:52,...,NaN,NaN,NaN,NaN,9,records100/00000/00008_lr,records500/00000/00008_hr,1,1.1; 1.5,ANORMAL
1,39,13619.0,56.0,0,NaN,NaN,2.0,0.0,CS-12 E,1985-02-16 12:54:01,...,NaN,NaN,NaN,NaN,9,records100/00000/00039_lr,records500/00000/00039_hr,1,1.1; 1.2; 1.5,ANORMAL
2,50,16961.0,59.0,1,NaN,64.0,2.0,0.0,CS-12 E,1985-05-09 12:01:24,...,NaN,NaN,NaN,NaN,7,records100/00000/00050_lr,records500/00000/00050_hr,1,1.1; 1.3,ANORMAL
3,63,15265.0,30.0,0,NaN,78.0,2.0,0.0,CS-12 E,1985-07-03 09:27:10,...,NaN,NaN,NaN,NaN,10,records100/00000/00063_lr,records500/00000/00063_hr,1,1.1; 1.5,ANORMAL
4,77,8555.0,43.0,0,NaN,63.0,0.0,0.0,CS-12 E,1985-09-27 09:55:22,...,NaN,NaN,NaN,NaN,6,records100/00000/00077_lr,records500/00000/00077_hr,1,1.1; 1.3,ANORMAL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5463,21820,18655.0,80.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-03-03 17:34:11,...,NaN,NaN,NaN,NaN,7,records100/21000/21820_lr,records500/21000/21820_hr,1,1.1,ANORMAL
5464,21824,19977.0,69.0,0,NaN,NaN,1.0,2.0,AT-60 3,2001-05-10 08:10:23,...,NaN,NaN,NaN,NaN,8,records100/21000/21824_lr,records500/21000/21824_hr,1,1.1; 1.5,ANORMAL
5465,21826,9178.0,82.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-13 18:20:52,...,NaN,NaN,NaN,NaN,10,records100/21000/21826_lr,records500/21000/21826_hr,1,1.1; 1.5,ANORMAL
5466,21827,13862.0,79.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-20 16:30:53,...,NaN,NaN,"4ES,SVES",NaN,5,records100/21000/21827_lr,records500/21000/21827_hr,1,1.1; 1.2; 1.5; 1.6,ANORMAL


### Alocando os Reespctivos registros nas suas reeespectivas labels e sublabels(ECG-Organizado)

In [7]:
import pandas as pd
import shutil
from pathlib import Path

# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

PASTA = Path.cwd()

BASE_PTBXL = PASTA / "PTB-XL"
CLASSIFICACAO = PASTA / "resultados" / "Classificacao"
SAIDA = PASTA / "resultados" / "ECG_Organizados"

SUBCATEGORIAS = {
    "1.1": "MI",
    "1.2": "STTC",
    "1.3": "CD",
    "1.4": "HYP",
    "1.5": "FORM",
    "1.6": "RHYTHM"
}

# ============================================================
# 2. CARREGAR AS 445 INSTÂNCIAS EXCLUÍDAS
# ============================================================

arquivos_exclusao = list(
    (PASTA / "resultados").glob(
        "instancias_NORM_e_anormal_juntos*.csv"
    )
)

if len(arquivos_exclusao) != 1:
    raise RuntimeError(
        "É necessário encontrar exatamente um CSV de exclusões "
        "na pasta resultados."
    )

exclusoes = pd.read_csv(arquivos_exclusao[0])

ids_excluidos = set(
    exclusoes["ecg_id"].astype(int)
)

assert len(ids_excluidos) == 445

# ============================================================
# 3. ORGANIZAR OS REGISTROS
# ============================================================

def organizar(nome_classe, arquivo_csv, destino):

    df = pd.read_csv(arquivo_csv)

    # Garantir que nenhuma instância excluída entre
    ids = set(df["ecg_id"].astype(int))

    conflitantes = ids & ids_excluidos

    if conflitantes:
        raise ValueError(
            f"{nome_classe} contém instâncias excluídas: "
            f"{sorted(conflitantes)}"
        )

    pasta_registros = destino / "registros"
    pasta_registros.mkdir(parents=True, exist_ok=True)

    registros_exportados = []
    ausentes = []

    for _, instancia in df.iterrows():

        ecg_id = int(instancia["ecg_id"])

        # Exemplo:
        # records500/00000/00004_hr
        caminho_relativo = Path(instancia["filename_hr"])

        origem_base = BASE_PTBXL / caminho_relativo

        origem_dat = origem_base.with_suffix(".dat")
        origem_hea = origem_base.with_suffix(".hea")

        if not origem_dat.exists() or not origem_hea.exists():

            ausentes.append(ecg_id)
            continue

        # Mantém a estrutura relativa de diretórios dentro
        # de registros para preservar possíveis referências.
        destino_base = pasta_registros / caminho_relativo

        destino_base.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        destino_dat = destino_base.with_suffix(".dat")
        destino_hea = destino_base.with_suffix(".hea")

        shutil.copy2(origem_dat, destino_dat)
        shutil.copy2(origem_hea, destino_hea)

        registro = instancia.to_dict()

        registro["arquivo_dat"] = str(
            destino_dat.relative_to(destino)
        )

        registro["arquivo_hea"] = str(
            destino_hea.relative_to(destino)
        )

        registros_exportados.append(registro)

    # CSV apenas com os registros efetivamente copiados
    df_saida = pd.DataFrame(
        registros_exportados,
        columns=list(df.columns) + [
            "arquivo_dat",
            "arquivo_hea"
        ]
    )

    df_saida.to_csv(
        destino / "instancias.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"{nome_classe}: "
        f"{len(df_saida)} copiados | "
        f"{len(ausentes)} ausentes"
    )

    return ausentes


# ============================================================
# 4. CLASSE 0
# ============================================================

todos_ausentes = {}

todos_ausentes["0"] = organizar(
    "0 - Normal/Sinusal",
    CLASSIFICACAO / "0" / "classe_0.csv",
    SAIDA / "0"
)

# ============================================================
# 5. SUBCATEGORIAS DA CLASSE 1
# ============================================================

for numero, nome in SUBCATEGORIAS.items():

    arquivo = (
        CLASSIFICACAO
        / "1"
        / numero
        / f"classe_{numero}_{nome}.csv"
    )

    destino = SAIDA / "1" / f"{numero}_{nome}"

    todos_ausentes[numero] = organizar(
        f"{numero} - {nome}",
        arquivo,
        destino
    )

# ============================================================
# 6. RELATÓRIO FINAL
# ============================================================

print("\n" + "=" * 55)
print("ORGANIZAÇÃO CONCLUÍDA")
print("=" * 55)

print("IDs conflitantes bloqueados:", len(ids_excluidos))
print("Pasta de saída:", SAIDA)

total_ausentes = sum(
    len(ids) for ids in todos_ausentes.values()
)

print("Ocorrências de arquivos ausentes:", total_ausentes)

if total_ausentes:
    print(
        "Atenção: alguns registros não foram copiados. "
        "Confira BASE_PTBXL e os caminhos filename_hr."
    )

0 - Normal/Sinusal: 8069 copiados | 0 ausentes
1.1 - MI: 5468 copiados | 0 ausentes
1.2 - STTC: 5202 copiados | 0 ausentes
1.3 - CD: 4483 copiados | 0 ausentes
1.4 - HYP: 2644 copiados | 0 ausentes
1.5 - FORM: 7233 copiados | 0 ausentes
1.6 - RHYTHM: 2136 copiados | 0 ausentes

ORGANIZAÇÃO CONCLUÍDA
IDs conflitantes bloqueados: 445
Pasta de saída: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\resultados\ECG_Organizados
Ocorrências de arquivos ausentes: 0
